# Anomaly Detection — IsolationForest Training & Evaluation

**Goal:** Train an IsolationForest model for anomaly detection and compare against an IQR-based statistical baseline.

**Baseline:** IQR (Interquartile Range) rule-based outlier detection — no training required  
**After:** Trained `IsolationForest` (100 estimators, contamination tuned empirically)

**Dataset:** Synthetic tabular dataset with injected 5% anomalies (fully reproducible, no download needed)  
_The same approach is used with KDD Cup '99 / NSL-KDD in the production system._

In [ ]:
import json
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

os.makedirs('../evaluation', exist_ok=True)
os.makedirs('../data/raw',   exist_ok=True)

print('Libraries loaded successfully.')

## 1. Generate Anomaly Detection Dataset

We create a realistic tabular dataset simulating a business/network scenario:  
- 5,000 rows, 15 features, 5% anomaly rate (250 anomalous points)
- Normal data: multivariate Gaussian clusters
- Anomalies: shifted, high-variance outlier points injected deliberately

In [ ]:
N_SAMPLES    = 5000
N_FEATURES   = 15
ANOMALY_RATE = 0.05
N_ANOMALIES  = int(N_SAMPLES * ANOMALY_RATE)  # 250

rng = np.random.default_rng(SEED)

# Normal data: 3 Gaussian clusters
c1 = rng.multivariate_normal(mean=np.zeros(N_FEATURES),
                              cov=np.eye(N_FEATURES),
                              size=N_SAMPLES // 3)
c2 = rng.multivariate_normal(mean=np.ones(N_FEATURES) * 3,
                              cov=np.eye(N_FEATURES) * 0.5,
                              size=N_SAMPLES // 3)
c3 = rng.multivariate_normal(mean=np.ones(N_FEATURES) * -2,
                              cov=np.eye(N_FEATURES) * 1.5,
                              size=N_SAMPLES - 2 * (N_SAMPLES // 3))

X_normal = np.vstack([c1, c2, c3])
y_normal  = np.zeros(N_SAMPLES, dtype=int)

# Anomalies: distant, high-variance points
X_anomaly = rng.uniform(low=8, high=15, size=(N_ANOMALIES, N_FEATURES))
X_anomaly *= rng.choice([-1, 1], size=X_anomaly.shape)  # random sign flip
y_anomaly  = np.ones(N_ANOMALIES, dtype=int)

# Combine and shuffle
X_all = np.vstack([X_normal, X_anomaly])
y_all = np.concatenate([y_normal, y_anomaly])
shuffle_idx = rng.permutation(len(y_all))
X_all, y_all = X_all[shuffle_idx], y_all[shuffle_idx]

# Save dataset
col_names = [f'feature_{i+1}' for i in range(N_FEATURES)] + ['is_anomaly']
df_anomaly = pd.DataFrame(np.column_stack([X_all, y_all]), columns=col_names)
df_anomaly['is_anomaly'] = df_anomaly['is_anomaly'].astype(int)
df_anomaly.to_csv('../data/raw/anomaly_dataset.csv', index=False)

print(f'Dataset shape : {X_all.shape}')
print(f'Normal rows   : {(y_all == 0).sum()} ({(y_all == 0).mean()*100:.1f}%)')
print(f'Anomaly rows  : {(y_all == 1).sum()} ({(y_all == 1).mean()*100:.1f}%)')

## 2. Train / Test Split + Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)

scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train_s.shape} | Test: {X_test_s.shape}')
print(f'Test anomaly rate: {y_test.mean()*100:.1f}%')

## 3. Baseline — IQR-Based Outlier Detection

In [ ]:
def iqr_anomaly_score(X_train, X_test):
    """Flag a point as anomaly if ANY feature exceeds Q3 + 1.5*IQR or Q1 - 1.5*IQR."""
    Q1  = np.percentile(X_train, 25, axis=0)
    Q3  = np.percentile(X_train, 75, axis=0)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    # Score = max normalized deviation across features
    dev   = np.maximum(X_test - upper, lower - X_test)  # >0 means outlier
    score = dev.max(axis=1)
    pred  = (score > 0).astype(int)
    return pred, score

y_pred_iqr, score_iqr = iqr_anomaly_score(X_train_s, X_test_s)

iqr_precision = precision_score(y_test, y_pred_iqr,  zero_division=0)
iqr_recall    = recall_score(y_test,    y_pred_iqr,  zero_division=0)
iqr_f1        = f1_score(y_test,        y_pred_iqr,  zero_division=0)
iqr_auc       = roc_auc_score(y_test,   score_iqr)
iqr_fp_rate   = ((y_pred_iqr == 1) & (y_test == 0)).sum() / (y_test == 0).sum()

print('IQR Baseline Results:')
print(f'  Precision : {iqr_precision:.4f}')
print(f'  Recall    : {iqr_recall:.4f}')
print(f'  F1 Score  : {iqr_f1:.4f}')
print(f'  ROC-AUC   : {iqr_auc:.4f}')
print(f'  FP Rate   : {iqr_fp_rate:.4f}')

## 4. Train IsolationForest

In [ ]:
iso = IsolationForest(
    n_estimators=100,
    contamination=ANOMALY_RATE,  # 0.05
    max_samples='auto',
    random_state=SEED,
    n_jobs=-1
)

# IsolationForest is UNSUPERVISED — fits on X_train only (no labels used)
iso.fit(X_train_s)
print('IsolationForest trained.')

# Predict: -1 = anomaly, +1 = normal
y_pred_if_raw  = iso.predict(X_test_s)
y_pred_if      = (y_pred_if_raw == -1).astype(int)
score_if       = -iso.score_samples(X_test_s)  # negate: higher = more anomalous

if_precision = precision_score(y_test, y_pred_if, zero_division=0)
if_recall    = recall_score(y_test,    y_pred_if, zero_division=0)
if_f1        = f1_score(y_test,        y_pred_if, zero_division=0)
if_auc       = roc_auc_score(y_test,   score_if)
if_fp_rate   = ((y_pred_if == 1) & (y_test == 0)).sum() / (y_test == 0).sum()

print('\nIsolationForest Results:')
print(f'  Precision : {if_precision:.4f}')
print(f'  Recall    : {if_recall:.4f}')
print(f'  F1 Score  : {if_f1:.4f}')
print(f'  ROC-AUC   : {if_auc:.4f}')
print(f'  FP Rate   : {if_fp_rate:.4f}')

## 5. Results Table — Before vs After

In [ ]:
df_results = pd.DataFrame({
    'Metric':            ['Precision', 'Recall', 'F1 Score', 'ROC-AUC', 'False Positive Rate'],
    'IQR Baseline':      [f'{iqr_precision:.3f}', f'{iqr_recall:.3f}', f'{iqr_f1:.3f}',
                          f'{iqr_auc:.3f}', f'{iqr_fp_rate:.3f}'],
    'IsolationForest':   [f'{if_precision:.3f}',  f'{if_recall:.3f}',  f'{if_f1:.3f}',
                          f'{if_auc:.3f}',  f'{if_fp_rate:.3f}'],
    'Improvement':       [
        f'+{(if_precision - iqr_precision)*100:+.1f}pp' if if_precision >= iqr_precision else f'{(if_precision - iqr_precision)*100:.1f}pp',
        f'{(if_recall    - iqr_recall)    *100:+.1f}pp',
        f'{(if_f1        - iqr_f1)        *100:+.1f}pp',
        f'{(if_auc       - iqr_auc)       *100:+.1f}pp',
        f'{(if_fp_rate   - iqr_fp_rate)   *100:+.1f}pp'
    ]
})
print(df_results.to_string(index=False))

# Save JSON
results_if = {
    'baseline': {
        'method': 'IQR Rule-Based',
        'precision': round(iqr_precision, 4), 'recall': round(iqr_recall, 4),
        'f1': round(iqr_f1, 4), 'roc_auc': round(iqr_auc, 4), 'fp_rate': round(iqr_fp_rate, 4)
    },
    'isolation_forest': {
        'method': 'IsolationForest (n_estimators=100, contamination=0.05)',
        'precision': round(if_precision, 4), 'recall': round(if_recall, 4),
        'f1': round(if_f1, 4), 'roc_auc': round(if_auc, 4), 'fp_rate': round(if_fp_rate, 4)
    },
    'improvement': {
        'f1_pp':  round((if_f1  - iqr_f1)  * 100, 2),
        'auc_pp': round((if_auc - iqr_auc) * 100, 2),
    }
}
with open('../evaluation/isolation_forest_results.json', 'w') as f:
    json.dump(results_if, f, indent=2)
print('\nResults saved to ../evaluation/isolation_forest_results.json')

## 6. Contamination Parameter Sweep

In [ ]:
contamination_values = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20]
sweep_f1, sweep_prec, sweep_rec, sweep_auc = [], [], [], []

for c in contamination_values:
    clf = IsolationForest(n_estimators=100, contamination=c, random_state=SEED, n_jobs=-1)
    clf.fit(X_train_s)
    yp  = (clf.predict(X_test_s) == -1).astype(int)
    sc  = -clf.score_samples(X_test_s)
    sweep_f1.append(f1_score(y_test, yp, zero_division=0))
    sweep_prec.append(precision_score(y_test, yp, zero_division=0))
    sweep_rec.append(recall_score(y_test, yp, zero_division=0))
    sweep_auc.append(roc_auc_score(y_test, sc))

print('Contamination Sweep Results:')
df_sweep = pd.DataFrame({
    'Contamination': contamination_values,
    'Precision': [f'{v:.3f}' for v in sweep_prec],
    'Recall':    [f'{v:.3f}' for v in sweep_rec],
    'F1':        [f'{v:.3f}' for v in sweep_f1],
    'ROC-AUC':   [f'{v:.3f}' for v in sweep_auc],
})
print(df_sweep.to_string(index=False))

best_c = contamination_values[np.argmax(sweep_f1)]
print(f'\nBest contamination by F1: {best_c}')

## 7. Plots

In [ ]:
# ── ROC Curves ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for name, scores in [('IQR Baseline', score_iqr), ('IsolationForest', score_if)]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — IQR vs IsolationForest', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../evaluation/if_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: if_roc_curve.png')

In [ ]:
# ── Precision-Recall Curves ───────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for name, scores in [('IQR Baseline', score_iqr), ('IsolationForest', score_if)]:
    prec, rec, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    ax.plot(rec, prec, lw=2, label=f'{name} (AP = {ap:.3f})')

ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../evaluation/if_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: if_precision_recall.png')

In [ ]:
# ── Anomaly Score Distribution ────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(score_if[y_test == 0], bins=60, alpha=0.6, color='#4ade80', label='Normal',  density=True)
ax.hist(score_if[y_test == 1], bins=60, alpha=0.7, color='#f87171', label='Anomaly', density=True)

# Decision threshold (at contamination=5%)
threshold = np.percentile(score_if, 100 * (1 - ANOMALY_RATE))
ax.axvline(threshold, color='#6366f1', linestyle='--', lw=2,
           label=f'Decision threshold (contamination={ANOMALY_RATE})')

ax.set_xlabel('Anomaly Score (higher = more anomalous)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('IsolationForest Anomaly Score Distribution', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../evaluation/if_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: if_score_distribution.png')

In [ ]:
# ── Contamination Sweep ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(contamination_values, sweep_f1,  marker='o', lw=2, label='F1 Score',  color='#6366f1')
ax.plot(contamination_values, sweep_prec, marker='s', lw=2, label='Precision', color='#f59e0b')
ax.plot(contamination_values, sweep_rec,  marker='^', lw=2, label='Recall',    color='#10b981')
ax.axvline(ANOMALY_RATE, color='#ef4444', linestyle='--', lw=1.5,
           label=f'Selected value ({ANOMALY_RATE})')

ax.set_xlabel('Contamination Parameter', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('IsolationForest — Contamination Parameter Sweep', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../evaluation/if_contamination_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: if_contamination_sweep.png')

In [ ]:
# ── Confusion Matrix (IsolationForest) ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, yp, title in zip(
    axes,
    [y_pred_iqr, y_pred_if],
    ['IQR Baseline', 'IsolationForest']
):
    cm = confusion_matrix(y_test, yp)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anomaly'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrix — IQR vs IsolationForest', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../evaluation/if_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: if_confusion_matrices.png')

## 8. Final Summary

In [ ]:
print('=' * 58)
print('       ISOLATION FOREST RESULTS SUMMARY')
print('=' * 58)
print(f'  {"Metric":<22} {"IQR Baseline":>14} {"IsolationForest":>16}')
print('-' * 58)
rows = [
    ('Precision',          iqr_precision, if_precision),
    ('Recall',             iqr_recall,    if_recall),
    ('F1 Score',           iqr_f1,        if_f1),
    ('ROC-AUC',            iqr_auc,       if_auc),
    ('False Positive Rate', iqr_fp_rate,   if_fp_rate),
]
for metric, base, model in rows:
    diff = model - base
    arrow = '↑' if diff > 0 else '↓'
    print(f'  {metric:<22} {base:>14.3f} {model:>14.3f}  {arrow}{abs(diff)*100:.1f}pp')
print('=' * 58)
print(f'  F1 improvement : +{results_if["improvement"]["f1_pp"]:.1f} percentage points')
print(f'  AUC improvement: +{results_if["improvement"]["auc_pp"]:.1f} percentage points')
print('=' * 58)
print('  Plots saved in: ../evaluation/')
print('  JSON  saved in: ../evaluation/isolation_forest_results.json')